# 01 — Library Overview
## How a PADRE simulation is put together, and how to tell whether its answer is right

PADRE reads a plain-text **input deck**: a list of cards (`MESH`, `REGION`, `DOPING`, `SOLVE`, ...) executed
top to bottom. `nanohubpadre` gives each card a Python class, assembles the deck, runs PADRE and parses the
output files.

**Learning objectives**
1. Know what every card in a deck does and in which order the cards must appear.
2. Build a p-n diode by hand, card by card, and read the generated deck.
3. Build the same diode with a *device factory* and see what the factory adds.
4. Run PADRE, find its outputs, and load them as NumPy arrays.
5. Recognise the **six most common ways a simulation silently gives a wrong answer**, and check for each.

In [1]:
# ── Setup ────────────────────────────────────────────────────────────────
# PADRE is a compiled simulator. On nanoHUB it lives in an environment
# module that must be loaded into this Python kernel before the first
# simulation runs; elsewhere, PADRE just needs to be on your PATH.
import shutil
from nanohubpadre import use

if shutil.which("padre") is None:
    try:
        use("padre-2.4E-r15")          # nanoHUB module name
    except Exception as exc:           # not on nanoHUB: that is fine
        print("Could not load the PADRE module:", exc)

print("PADRE executable:", shutil.which("padre") or "NOT FOUND - simulations will fail")

PADRE executable: /apps/share64/debian10/padre/padre-2.4E/r15/bin/padre


In [2]:
# ── Physical constants: textbook values vs. the values PADRE really uses ──
# PADRE prints its constants with `MODELS ... PRINT`. A fair comparison of
# simulation against a formula must feed the formula PADRE's numbers:
# e.g. the textbook n_i = 1.5e10 cm^-3 alone changes every analytic diode
# current by (1.5e10 / 9.963e9)^2 = 2.3x before any physics is compared.
import numpy as np

Q      = 1.602e-19        # elementary charge (C)
KT     = 0.025851         # thermal voltage kT/q at 300 K (V)
EPS0   = 8.854e-14        # vacuum permittivity (F/cm)
EPS_SI = 11.8 * EPS0      # silicon permittivity in PADRE (F/cm)
EPS_OX = 3.9 * EPS0       # SiO2 permittivity (F/cm)
NI     = 9.963e9          # intrinsic carrier density of Si at 300 K (cm^-3)
EG     = 1.12             # band gap (eV)
CHI_SI = 4.17             # electron affinity of Si in PADRE (eV)
NC, NV = 3.2e19, 2.03e19  # effective densities of states (cm^-3)

# The values most textbooks quote (e.g. Sze & Ng, Pierret), for contrast.
TEXTBOOK = dict(NI=1.5e10, EPS_SI=11.7 * EPS0, CHI_SI=4.05, NC=2.8e19, NV=1.04e19)


def compare(rows, title="Textbook vs. PADRE"):
    """Print a comparison table.

    rows: list of (quantity, textbook_value, padre_value, unit, why_different)
    """
    print(title)
    print("-" * 125)
    print(f"{'quantity':<40}{'textbook':>12}{'PADRE':>12}{'diff':>9}   why they differ")
    print("-" * 125)
    for name, tb, pd, unit, why in rows:
        d = f"{(pd - tb) / tb * 100:>+8.1f}%" if tb else "     n/a "
        print(f"{name + ' (' + unit + ')':<40}{tb:>12.4g}{pd:>12.4g}{d}   {why}")
    print("-" * 125)


compare([
    ("intrinsic density n_i", TEXTBOOK["NI"], NI, "cm^-3",
     "modern value (Sproul & Green 1991); 1.5e10 is the older textbook number"),
    ("Si permittivity", TEXTBOOK["EPS_SI"], EPS_SI, "F/cm", "PADRE uses eps_r = 11.8, Sze uses 11.7"),
    ("electron affinity chi", TEXTBOOK["CHI_SI"], CHI_SI, "eV", "PADRE (like Silvaco Atlas) uses 4.17"),
    ("conduction-band DOS Nc", TEXTBOOK["NC"], NC, "cm^-3", "different effective-mass fit"),
    ("valence-band DOS Nv", TEXTBOOK["NV"], NV, "cm^-3", "different effective-mass fit"),
], title="Material constants")

Material constants
-----------------------------------------------------------------------------------------------------------------------------
quantity                                    textbook       PADRE     diff   why they differ
-----------------------------------------------------------------------------------------------------------------------------
intrinsic density n_i (cm^-3)                1.5e+10   9.963e+09   -33.6%   modern value (Sproul & Green 1991); 1.5e10 is the older textbook number
Si permittivity (F/cm)                     1.036e-12   1.045e-12    +0.9%   PADRE uses eps_r = 11.8, Sze uses 11.7
electron affinity chi (eV)                      4.05        4.17    +3.0%   PADRE (like Silvaco Atlas) uses 4.17
conduction-band DOS Nc (cm^-3)               2.8e+19     3.2e+19   +14.3%   different effective-mass fit
valence-band DOS Nv (cm^-3)                 1.04e+19    2.03e+19   +95.2%   different effective-mass fit
---------------------------------------------------

---
## 1. Anatomy of a deck

```
TITLE     PN diode                       <- free text
MESH      rect nx=.. ny=..               <- the grid: node counts
X.MESH / Y.MESH  node location ratio      <- where the nodes are (graded spacing)
REGION    ix.l ix.h iy.l iy.h silicon     <- which nodes are which material
ELECTRODE num ix.l ix.h iy.l iy.h         <- which nodes are contacts
DOPING    uniform|gaussian ...            <- the impurity profile
CONTACT   neutral | workfunction= ...     <- boundary condition at each electrode
MATERIAL  taun0= taup0= ...               <- material parameters (lifetimes, mobility, ...)
MODELS    srh conmob fldmob ...           <- which physics is switched on
SYSTEM    electrons holes newton          <- which equations, which solver
SOLVE     init                            <- equilibrium first, always
LOG       outf=iv                         <- start recording terminal I-V
SOLVE     prev/proj v1= vstep= nsteps=    <- bias steps
PLOT.1D / PLOT.3D                         <- dump internal quantities along a line / everywhere
END
```

Structure cards (`MESH` ... `SYSTEM`) describe **the device**. `SOLVE`, `LOG` and `PLOT` cards describe
**the experiment**, and they are executed in order, like a lab procedure.

---
## 2. Building a diode by hand

We build a 2 µm long, 0.5 µm tall silicon diode: p-type 1e17 cm⁻³ on the left half, n-type 1e17 cm⁻³ on the right half,
with ohmic contacts at both ends.

### 2.1 Mesh
`X.MESH node=k location=x ratio=r` places node *k* at position *x*. Between two anchor nodes, spacing grows
geometrically by `ratio` per interval. Values below 1 make cells shrink **towards** the later anchor. We shrink the
cells toward the junction at x = 1 µm, where carrier densities change by ten orders of magnitude in about 100 nm.

In [3]:
from nanohubpadre import (Simulation, Mesh, Region, Electrode, Doping, Contact,
                          Material, Models, System, Solve, Log, Plot1D)

mesh = Mesh(nx=100, ny=3)
mesh.add_x_mesh(1, 0.0)                 # node 1 at x = 0
mesh.add_x_mesh(50, 1.0, ratio=0.93)    # cells shrink toward the junction
mesh.add_x_mesh(100, 2.0, ratio=1.075)  # ...and grow again toward the right contact
mesh.add_y_mesh(1, 0.0)
mesh.add_y_mesh(3, 0.5)                 # 3 rows: the device is effectively 1-D

print(f"{mesh.nx} x {mesh.ny} = {mesh.nx * mesh.ny} nodes")

100 x 3 = 300 nodes


> **Node limit.** PADRE on nanoHUB stops with *"Insufficient storage for real numerical factorization"* above
> roughly 2500 nodes. The factories warn you before that happens; hand-built decks do not.

### 2.2 Region, electrodes, doping, contacts
Regions and electrodes are given in **node indices** (ix, iy); doping boxes in **micrometres**.

In [4]:
sim = Simulation(title="Hand-built PN diode")
sim.mesh = mesh

# every node is silicon
sim.add_region(Region(1, ix_low=1, ix_high=100, iy_low=1, iy_high=3, silicon=True))

# electrode 1 = left column (the p side, the anode); electrode 2 = right column (the cathode)
sim.add_electrode(Electrode(1, ix_low=1, ix_high=1, iy_low=1, iy_high=3))
sim.add_electrode(Electrode(2, ix_low=100, ix_high=100, iy_low=1, iy_high=3))

# doping boxes in micrometres
sim.add_doping(Doping(uniform=True, p_type=True, concentration=1e17,
                      x_left=0.0, x_right=1.0, y_top=0.0, y_bottom=0.5))
sim.add_doping(Doping(uniform=True, n_type=True, concentration=1e17,
                      x_left=1.0, x_right=2.0, y_top=0.0, y_bottom=0.5))

# "neutral" = ideal ohmic contact: carrier densities pinned to their equilibrium values
sim.add_contact(Contact(all_contacts=True, neutral=True))

<Simulation title='Hand-built PN diode', regions=1, electrodes=2, dopings=2, commands=0>

### 2.3 Material, models and solver

* `MATERIAL taun0 taup0` sets the SRH lifetimes (s). With 1 µs, the diffusion length $L=\sqrt{D\tau}\approx 44$ µm,
  much longer than the device, so this is a **short-base** diode. Notebook 02 shows why that matters.
* `MODELS`: `srh` recombination and `conmob` doping-dependent mobility. We leave `fldmob` off here; notebook 02
  discusses how it interacts with diffusion currents.
* `SYSTEM electrons holes newton`: solve Poisson + both continuity equations, fully coupled.

In [5]:
sim.add_material(Material(name="silicon", taun0=1e-6, taup0=1e-6))
sim.models = Models(temperature=300, srh=True, conmob=True)
sim.system = System(electrons=True, holes=True, newton=True)

### 2.4 The experiment: solve sequence

1. `SOLVE init` — equilibrium, from a charge-neutral initial guess. **Always first.**
2. `LOG outf=iv` — from now on, append terminal voltages and currents to the file `iv`. Placing `LOG` *after* the
   equilibrium solve keeps that point out of the I-V file.
3. A **forward** sweep. The p side is electrode 1, so forward bias means **positive voltage on electrode 1**.
   The first bias step must use `PREV`: `PROJ` extrapolates from *two* earlier solutions, and after `init` there is only one.

In [6]:
sim.add_solve(Solve(initial=True, outfile="eq"))
sim.add_log(Log(ivfile="iv"))
sim.add_solve(Solve(previous=True, v1=0.0, vstep=0.05, nsteps=14, electrode=1,
                    no_append=True, outfile="fwd"))

print(sim.generate_deck())

  TITLE  Hand-built PN diode
mesh rect nx=100 ny=3
x.m n=1 l=0
x.m n=50 l=1.0 r=0.93
x.m n=100 l=2.0 r=1.075
y.m n=1 l=0
y.m n=3 l=0.5
region num=1 ix.l=1 ix.h=100 iy.l=1 iy.h=3 silicon
elec num=1 ix.l=1 ix.h=1 iy.l=1 iy.h=3
elec num=2 ix.l=100 ix.h=100 iy.l=1 iy.h=3
dop uniform p.type conc=1.000000e17 x.l=0 x.r=1.0 y.top=0 y.bot=0.5
dop uniform n.type conc=1.000000e17 x.l=1.0 x.r=2.0 y.top=0 y.bot=0.5
contact all neutral
material name=silicon taun0=1e-06 taup0=1e-06
models srh conmob temp=300
system electrons holes newton
solve init outf=eq
log outf=iv
solve prev no.append v1=0 vstep=0.05 nsteps=14 elect=1 outf=fwd

end


Every line of that deck now corresponds to a decision you made. Next, run it and read the results.
`run()` creates a working directory (named from a hash of the deck, so re-running an identical deck reuses
the cached result), writes the deck, runs PADRE there, and indexes every output file.

In [7]:
result = sim.run()
print("return code:", result.returncode)
print("working directory:", sim.working_dir)
print(sim.outputs.summary())

return code: 0
working directory: /tmp/nbrun_padre/01_Library_Overview/handbuilt_pn_diode_ee09368a061c8fb6261805c3f566bd5b
Simulation Outputs:
----------------------------------------

SOLUTION:
  - eq [loaded]
  - fwd [loaded]

IV_DATA:
  - iv [loaded]


In [8]:
iv = sim.get_iv_data()            # parse the LOG file ("iv")
v = iv.get_voltages(1)            # voltage on the swept electrode (the anode)
i = np.abs(iv.get_currents(1))    # current into the anode, A per um of depth

for vk, ik in zip(v, i):
    print(f"V = {vk:5.2f} V    I = {ik:10.3e} A/um")

V =  0.00 V    I =  1.273e-19 A/um
V =  0.05 V    I =  7.330e-18 A/um
V =  0.10 V    I =  3.148e-17 A/um
V =  0.15 V    I =  1.364e-16 A/um
V =  0.20 V    I =  6.917e-16 A/um
V =  0.25 V    I =  4.063e-15 A/um
V =  0.30 V    I =  2.604e-14 A/um
V =  0.35 V    I =  1.741e-13 A/um
V =  0.40 V    I =  1.185e-12 A/um
V =  0.45 V    I =  8.126e-12 A/um
V =  0.50 V    I =  5.588e-11 A/um
V =  0.55 V    I =  3.846e-10 A/um
V =  0.60 V    I =  2.647e-09 A/um
V =  0.65 V    I =  1.818e-08 A/um
V =  0.70 V    I =  1.238e-07 A/um


---
## 3. The same device from a factory

`create_pn_diode()` produces a diode like the one above in one call, and adds things that are easy to get wrong by hand:
a mesh window sized from the depletion width, PREV/PROJ sequencing, `LOG` placement, stepped ramps back to
0 V before a reverse sweep, warnings for bad inputs, and consistent output-file names.

In [9]:
from nanohubpadre import create_pn_diode

sim_f = create_pn_diode(length=2.0, p_doping=1e17, n_doping=1e17, fldmob=False,
                        log_iv=True, forward_sweep=(0.0, 0.7, 0.05))
print(sim_f.generate_deck())

  TITLE  PN Junction Diode
mesh rect nx=200 ny=3 width=1.0 outf=mesh
x.m n=1 l=0 r=1
x.m n=31 l=0.8547575619 r=0.8719
x.m n=101 l=1.0 r=1
x.m n=170 l=1.1452424381 r=1
x.m n=200 l=2.0 r=1.1469
y.m n=1 l=0 r=1
y.m n=3 l=1.0 r=1
region num=1 ix.l=1 ix.h=101 iy.l=1 iy.h=3 silicon
region num=1 ix.l=101 ix.h=200 iy.l=1 iy.h=3 silicon
elec num=1 ix.l=1 ix.h=1 iy.l=1 iy.h=3
elec num=2 ix.l=200 ix.h=200 iy.l=1 iy.h=3
dop uniform p.type conc=1.000000e17 reg=1 x.l=0 x.r=1.0 y.top=0
+     y.bot=1.0
dop uniform n.type conc=1.000000e17 reg=1 x.l=1.0 x.r=2.0 y.top=0
+     y.bot=1.0
contact all neutral
material name=silicon taun0=1e-06 taup0=1e-06 trap.typ=0 etrap=0
models srh conmob temp=300
system electrons holes newton
log outf=iv
solve init outf=eq
solve prev no.append v1=0 v2=0 vstep=0.05 nsteps=14 elect=1 outf=fwd

end


In [10]:
sim_f.run()
iv_f = sim_f.get_iv_data()
vf, i_f = iv_f.get_voltages(1), np.abs(iv_f.get_currents(1))

# Raw currents first: A per um of depth
for V in (0.3, 0.5, 0.7):
    a = i[np.argmin(abs(v - V))]
    b = i_f[np.argmin(abs(vf - V))]
    print(f"V = {V:.1f} V   hand-built {a:.3e}   factory {b:.3e}   ratio {a / b:.3f}")

V = 0.3 V   hand-built 2.604e-14   factory 5.207e-14   ratio 0.500
V = 0.5 V   hand-built 5.588e-11   factory 1.117e-10   ratio 0.500
V = 0.7 V   hand-built 1.238e-07   factory 2.475e-07   ratio 0.500


**The currents differ by exactly a factor of 2 — why?** Look at the two decks: our hand-built diode is
**0.5 µm tall** (`Y.MESH ... 0.5`), the factory diode is **1 µm tall**. PADRE's currents are "per µm of depth"
(the z direction, into the page), but they still scale with the *height* of the cross-section, because the
current flows through the whole height. The junction area of the hand-built device is half as large. Dividing by
the cross-section area turns both into a **current density**, which is what device physics is really about:

In [11]:
A_hand = 0.5e-4 * 1e-4          # 0.5 um tall x 1 um deep, in cm^2
A_fact = 1.0e-4 * 1e-4          # 1 um tall x 1 um deep
for V in (0.3, 0.5, 0.7):
    a = i[np.argmin(abs(v - V))] / A_hand
    b = i_f[np.argmin(abs(vf - V))] / A_fact
    print(f"V = {V:.1f} V   hand-built {a:.4e} A/cm^2   factory {b:.4e} A/cm^2   ratio {a / b:.4f}")

V = 0.3 V   hand-built 5.2088e-06 A/cm^2   factory 5.2066e-06 A/cm^2   ratio 1.0004
V = 0.5 V   hand-built 1.1175e-02 A/cm^2   factory 1.1174e-02 A/cm^2   ratio 1.0001
V = 0.7 V   hand-built 2.4753e+01 A/cm^2   factory 2.4752e+01 A/cm^2   ratio 1.0001


Once normalised, two independently built meshes agree to better than a percent: a first sign that neither result
is a mesh artefact. Notebook 09 does a proper mesh-convergence study. **Lesson:** always check what area or depth a
simulated current refers to before comparing it with a formula or with another simulation.

`Simulation.describe()` lists every factory parameter with its default and unit:

In [12]:
from nanohubpadre import Simulation
Simulation.describe("pn_diode")

Parameter,Type,Default,Unit,Description
length,float,1.0,um,Total device length
width,float,1.0,um,Device width (y extent)
device_z_width,float,1.0,um,Depth in z; terminal currents scale with it
junction_position,float,0.5,,Junction position as fraction of length (0-1)
Parameter,Type,Default,Unit,Description
nx,int,200,,Mesh points in x direction
ny,int,3,,Mesh points in y direction
Parameter,Type,Default,Unit,Description
p_doping,float,1e+17,cm^-3,P-type doping concentration
n_doping,float,1e+17,cm^-3,N-type doping concentration


---
## 4. Six ways a simulation silently gives a wrong answer

A simulator almost always returns *a* number. These are the checks that tell you whether to believe it.
Each one below caused a real, silent error during the development of these tutorials.

### Mistake 1 — biasing the wrong electrode
A "forward sweep" of positive voltage on the **n** side reverse-biases the diode. The I-V file then holds only
leakage current, and it looks like a very bad diode. The factory warns you:

In [13]:
import warnings
with warnings.catch_warnings(record=True) as w:
    warnings.simplefilter("always")
    create_pn_diode(log_iv=True, forward_sweep=(0.0, 0.7, 0.05), sweep_electrode=2)
for x in w:
    print("WARNING:", str(x.message)[:200], "...")

### Mistake 2 — `PROJ` without two previous solutions
`SOLVE proj` right after `SOLVE init` asks PADRE to extrapolate from a solution that does not exist.
`generate_deck()` checks the sequence and warns:

In [14]:
bad = Simulation(title="bad sequencing")
bad.mesh = mesh
bad.add_region(Region(1, ix_low=1, ix_high=100, iy_low=1, iy_high=3, silicon=True))
bad.add_electrode(Electrode(1, ix_low=1, ix_high=1, iy_low=1, iy_high=3))
bad.add_electrode(Electrode(2, ix_low=100, ix_high=100, iy_low=1, iy_high=3))
bad.add_solve(Solve(initial=True))
bad.add_solve(Solve(project=True, v1=0.0, vstep=0.1, nsteps=5, electrode=1))
with warnings.catch_warnings(record=True) as w:
    warnings.simplefilter("always")
    bad.generate_deck()
for x in w:
    print("WARNING:", x.message)

### Mistake 3 — an output file overwriting another
PADRE names the solution file of each bias point by **incrementing the last character** of `outf=`:
`idvd`, `idve`, `idvf`, `idvg`, ... If one of those names equals your I-V log file, PADRE overwrites the log
partway through the sweep and the first points silently disappear. The factories pick safe names; here is
the naming rule, so you can avoid it in hand-built decks:

In [15]:
from nanohubpadre.devices._common import padre_outfile_names, safe_sweep_outfile
print("outf=idvd writes:", padre_outfile_names("idvd", 6))
print("a sweep that must not overwrite 'idvg' should use outf =", safe_sweep_outfile("idvd", 30, ["idvg"]))

outf=idvd writes: ['idvd', 'idve', 'idvf', 'idvg', 'idvh', 'idvi']
a sweep that must not overwrite 'idvg' should use outf = sidvd


### Mistake 4 — trusting currents below the numerical noise floor
Terminal currents must obey Kirchhoff's law: for a two-terminal device $I_1 + I_2 = 0$ **exactly**.
When a current is so small that it is buried in round-off, this balance fails. `IVData.check_continuity()` measures
$|I_1+I_2|/\max(|I_1|,|I_2|)$ at every point. Never fit an ideality factor or saturation current to points
above ~1 %.

In [16]:
err = iv_f.continuity_error()
for vk, ik, ek in zip(vf, i_f, err):
    flag = "  <-- noise floor, do not use" if ek > 0.01 else ""
    print(f"V = {vk:5.2f}  I = {ik:9.2e}  |I1+I2|/|I| = {ek:8.1e}{flag}")

V =  0.00  I =  1.65e-19  |I1+I2|/|I| =  2.0e-01  <-- noise floor, do not use
V =  0.00  I =  1.65e-19  |I1+I2|/|I| =  2.0e-01  <-- noise floor, do not use
V =  0.05  I =  1.44e-17  |I1+I2|/|I| =  2.7e-02  <-- noise floor, do not use
V =  0.10  I =  6.29e-17  |I1+I2|/|I| =  6.8e-03
V =  0.15  I =  2.72e-16  |I1+I2|/|I| =  4.6e-04
V =  0.20  I =  1.38e-15  |I1+I2|/|I| =  3.6e-04
V =  0.25  I =  8.12e-15  |I1+I2|/|I| =  3.2e-05
V =  0.30  I =  5.21e-14  |I1+I2|/|I| =  6.7e-08
V =  0.35  I =  3.48e-13  |I1+I2|/|I| =  1.2e-06
V =  0.40  I =  2.37e-12  |I1+I2|/|I| =  1.1e-07
V =  0.45  I =  1.62e-11  |I1+I2|/|I| =  1.8e-08
V =  0.50  I =  1.12e-10  |I1+I2|/|I| =  2.7e-08
V =  0.55  I =  7.69e-10  |I1+I2|/|I| =  1.3e-09
V =  0.60  I =  5.29e-09  |I1+I2|/|I| =  0.0e+00
V =  0.65  I =  3.64e-08  |I1+I2|/|I| =  0.0e+00
V =  0.70  I =  2.48e-07  |I1+I2|/|I| =  0.0e+00


### Mistake 5 — PADRE stopping early but exiting "successfully"
Some convergence failures end PADRE with exit code 0 and no *Aborted* banner: *"Bias stack exceeded"*,
*"Bias trapping below tolerances"*, *"nonconvergence in idirac"*. The remaining solves never run, and their
I-V points are simply missing. `run()` looks for these messages, warns, and returns a non-zero code.
**Always check `result.returncode` and the number of points you got.**

### Mistake 6 — a mesh too coarse, or too fine
Too coarse: the depletion region is sampled by one or two nodes. Too fine: cells smaller than an atom make the
matrix ill-conditioned (an early MOS-capacitor mesh had 3.6 femtometre cells). Re-run with the node count halved
and doubled. A trustworthy result changes by much less than the effect you are studying.

---
## 5. Units reminder
Currents are **A per µm of device depth**; capacitances are **F per µm**. To compare with a formula written
per cm², multiply the formula by the simulated area, e.g. `length_um * 1e-4 * depth_um * 1e-4` cm² for a
contact that spans the full device.

---
## Summary

| You learned | Key point |
|---|---|
| Deck structure | device cards first, then the experiment (`SOLVE`, `LOG`, `PLOT`) in order |
| Hand-built vs factory | factories add safe meshing, sequencing and file naming |
| Reading results | `get_iv_data()`, `outputs.get(name)`, units of A/µm |
| Checking results | right electrode, PREV before PROJ, no file clashes, continuity, return code, mesh |

**Next →** [02 — PN Junction Diode](02_PN_Diode.ipynb)